# Session 1

## Introduction to Biological Networks and Knowledge Graphs

### General

This session is about representing biomedical knowledge as a **graph**, and about
the specific ways that representation misleads us if we are not careful.

We want to understand what a biological network is, what a **knowledge graph** is,
and — the part people usually skip — how the two differ. A co-expression network is
*computed* from measurements: correlate every gene against every other gene and keep
what clears a threshold. A knowledge graph is *read out of recorded facts*: somebody
looked at the evidence and wrote down that this gene is implicated in that disease.
They look identical when drawn and they fail in opposite ways, so telling them apart
is the first thing worth learning.

So we build **both**, over the same genes, and spend the rest of the session on what
each one can and cannot say — starting from clinical ICD-10 codes, recovering
information the graph appears not to have, and finally connecting it to the molecular
data used in Session 2.

The graph is deliberately small — 881 nodes — so that every result can be checked
by eye. Everything here scales to graphs of millions of nodes without changing.

The data are:

| File | Description |
|------|-------------|
| `/data/session-1-data/kg_nodes.csv` | Node table — `id`, `type` (gene / disease / icd10), `name`, `extra` |
| `/data/session-1-data/kg_edges.csv` | Edge table — `source`, `target`, `type`, `weight`, `evidence` |
| `/data/session-1-data/kg_evidence.csv` | The same gene–disease edges split by *kind* of evidence |
| `/data/session-1-data/icd10_map.csv` | MONDO disease id → ICD-10 code and label |
| `/data/session-1-data/coexpr_expression.csv.gz` | 500 patients × 737 genes of TCGA-BRCA expression |

Built from [Open Targets](https://platform.opentargets.org/) (CC0 1.0) and the
[MONDO Disease Ontology](https://mondo.monarchinitiative.org/) (CC BY 4.0) by
`data-prep/build_kg_data.py`. That script downloads ~1.1 GB and cuts it down to a
breast-cancer-focused subset small enough to read by eye: **881 nodes** (760 genes,
90 diseases, 31 ICD-10 codes) and **1,768 edges**; we do not need to run it.

Around breast cancer and its subtypes we add **24 comparison diseases** — other
cancers, autoimmune and inflammatory conditions, neurological and cardiometabolic
disease — so that "which diseases resemble each other?" has a real answer rather
than one big breast-cancer blob.

Genes are keyed by **Ensembl gene ID**, the same identifier space as the TCGA-BRCA
transcriptomics matrix in Session 2, so the two datasets join natively. The
expression file is that matrix, cut down to the genes that are already nodes in
the graph — which is what lets us build an inferred *and* a curated network over
exactly the same genes.

### Part 1 — What is a Network, and What is a Knowledge Graph?

#### 1.1 Graphs

A graph is a set of **nodes** and a set of **edges** between them. Degree, paths and
components all follow from those two sets. This part establishes the vocabulary,
using a diagram and a four-node toy example, and shows the one operation that makes
graphs worth the trouble: composing edges into a path produces an answer nobody
stored.

#### 1.2 Inferred versus curated

Two networks can look identical and mean opposite things.

An **inferred** network computes its edges from measurements. Correlate every gene
against every other gene, keep what clears a threshold. The edges are a statistical
claim about *our data*, and the network can contain relationships nobody has
described before.

A **curated knowledge graph** reads its edges out of a database of recorded facts.
It can only contain what somebody already knew.

The failure modes are opposite, and neither is fixable by better code. In an
inferred network a spurious edge is usually a confounder. In a knowledge graph a
*missing* edge usually means nobody has looked yet.

#### 1.3 What makes it a knowledge graph

Typed nodes, typed edges, and provenance. Ours has three node types (gene, disease,
icd10) and three edge types (`associated_with`, `is_a`, `maps_to`), with an
association score and evidence count on every gene–disease edge — a separate table
splitting every gene–disease edge by the *kind* of evidence behind it. That is the
"specialised" in *a knowledge graph is a specialised class of network*.

#### 1.4 Building one of each

We then build both networks over the same 737 genes and compare them.

**The same gene looks completely different in the two networks.** `BRCA1`
co-expresses with the proliferation programme — tubulin-adjacent mitotic genes
that have nothing to do with it — plus `PALB2` and `FANCD2`, which are genuine
repair partners the knowledge graph has no way to record. Meanwhile the curated
graph gives its hereditary breast–ovarian diseases. `TP53` is starker still: a
27-disease hub in the curated graph, and essentially invisible to co-expression
(strongest |r| = 0.18), because it is regulated by protein stability rather than
transcription. Neither absence means *unimportant*; they mean different things,
and knowing which is the skill.

**And the inferred network has a knob the curated one does not.** Sweeping the
correlation threshold from 0.3 to 0.8 takes the same data from 39,593 edges to
200. Nothing in the data says where to stop. The knowledge graph has exactly
1,768 edges whatever we think — which removes that problem and introduces the
opposite one: we cannot loosen the criteria to see more.

### Part 2 — Building a Knowledge Graph with NetworkX

#### 2.1 Construction

A knowledge graph is a loop over a node table and an edge table. We write it out
explicitly before switching to the helper, so that nothing is magic.

One subtlety gets its own treatment: `is_a` is genuinely directional, but we work
undirected because degree and components behave better that way. The direction is
preserved as an edge attribute — without it, climbing the ontology in Part 3 walks
downwards half the time and returns plausible wrong answers rather than errors.

#### 2.2 Shape

We read the graph's summary statistics and ask what each one is really telling us.
The density is low, the clustering coefficient is near zero (the graph is close to
bipartite), and the hubs are all diseases.

That last one is a **construction artefact**: the build kept the top 30 genes per
disease, so diseases were always going to be the hubs. The habit this part is
trying to build is asking, every time, whether a graph property came from the data
or from how the data was cut.

#### 2.3 Annotation sparsity

The most important idea in the session. A missing edge has three possible causes —
the relationship does not exist, nobody has studied it, or it was recorded against
a different term — and the graph cannot tell us which.

We meet a concrete case: basal-like breast carcinoma, one of the most studied
breast cancer subtypes, has **zero** gene associations here, because the evidence
was filed against the near-synonymous term triple-negative breast carcinoma.

### Part 3 — Practical: Querying the Knowledge Graph

#### 3.1 Starting from ICD-10

[ICD-10](https://icd.who.int/browse10/2019/en) is the vocabulary hospitals
actually use. Only 19 of our 90 diseases carry a code — and the gap is not spread
evenly: **3 of 63** breast subtypes against **16 of 27** other diseases.

The reason is not a bad data source. ICD-10 subdivides breast cancer
**anatomically** — by quadrant — so there is no code for "triple-negative" or
"luminal A", and there cannot be one. Morphology lives in a separate
classification entirely. This is a **granularity mismatch** between the clinical
and molecular vocabularies, and no alternative database fixes it.

Climbing the `is_a` hierarchy recovers a code for 82 of 90 — with the caveat that
an inherited code describes the ancestor, not the subtype we started from.

#### 3.2 Which diseases share genes?

Projecting the bipartite gene–disease graph onto diseases alone turns "how similar
are these two diseases?" into a structural question. **Breast and ovarian cancer
share BRCA1, BRCA2 and BRIP1** — hereditary breast-ovarian cancer syndrome,
recovered from graph structure without being told the two diseases were related.

**But they also "share" tubulins.** TUBB, TUBA1B and the topoisomerases show up in
exactly the same list, not because of shared biology but because both cancers are
treated with taxanes and anthracyclines. The overall association score cannot
distinguish the two — but the **evidence types** can. Filter to genetic and somatic
evidence and every tubulin disappears, leaving the real hereditary genes. The
advantage of carrying provenance is that we can ask *what kind of claim is this*,
not just *how strong is it*.

**The diseases group themselves.** Community detection on that causal graph then
recovers the clinical taxonomy — a cancer community, an autoimmune/inflammatory
community, a metabolic one — without ever being told those categories exist.
Structure nobody encoded turns out to be recoverable from structure somebody did.

#### 3.3 Connecting to the omics data

Finally, the bridge to Session 2. The TCGA matrix uses versioned Ensembl IDs
(`ENSG00000012048.23`) and Open Targets does not, so a naive join matches
**nothing at all** — not fewer things, nothing. An empty join is the *safe*
failure; a partial one is what reaches publication.

All five PAM50 subtypes predicted in Session 2 are disease nodes in this graph, so
a subtype prediction becomes an entry point rather than a label.

#### 3.4 Both networks at once

The closing section puts the co-expression network from Part 1 back alongside the
curated graph, and asks the question the session title actually poses: *can
molecular data strengthen the evidence for a gene–disease relationship?*

Take the 16 genes with causal evidence for breast cancer and ask what co-expresses
with them. Three things come back.

**Corroboration.** Ten of 120 causal gene pairs co-express — and the strongest are
`BRCA2`–`BRIP1`, `BARD1`–`BRIP1`, `BARD1`–`BRCA2`: the homologous-recombination
complex, recovered twice over. The graph knows because families were sequenced;
the matrix knows because they are transcribed together. Two independent routes,
one biology.

**A confound we had already removed.** The seven strongest partners carrying any
curated breast-cancer edge are `TOP2A`, `TYMS`, `TUBA1B`, `TUBA1C`, `TUBB`, `TOP1`
and `CDK6` — precisely the chemotherapy targets Section 3.2 just filtered out.
They come back because they are proliferation genes, not because the filtering was
wrong. Two sources agreeing is only evidence when their errors are independent, and
here they are not.

**And 184 candidates nothing can sort.** Genes that co-express with the causal set
and carry no curated breast-cancer edge at all. `BLM`, `EXO1`, `FANCD2`, `RAD51`,
`MSH6` — real genome-stability genes worth following up — sit interleaved with
`BUB1B`, `HMMR`, `KNL1` and `RRM2`, which are proliferation artefacts. No column
in the table separates them; that judgement needs biology.

None of the gaps or mismatches along the way are bugs. They are what working with
real biomedical knowledge is like, and recognising them is the skill.

### Where this leads

Session 3 builds LLM agents — tools, MCP and skills — and Session 4 turns them on
graphs like this one, queried from multi-omics profiles. Everything in this
session — the coverage gaps, the inherited codes, the
technically-correct-but-misleading edges, and the candidate list nothing in the
data can sort — is what those agents have to get right, and what we need to check
them against.